<a href="https://colab.research.google.com/github/chrisampiah/Content/blob/main/myelin_age_gap_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Myelin Brain Age Gap & Diagnosis Classifier
**Dataset:** Large dataset of infancy and early childhood brain MRIs (T1w and T2w)  
**Authors:** Akinci D'Antonoli et al. — Zenodo DOI: 10.5281/zenodo.8055666  

## Pipeline
```
Step 1 — Age Regression:   MRI (T1w + T2w) → age_pred
Step 2 — Gap Computation:  gap = age_pred − age
Step 3 — 3-class diagnosis (subjects where gap ≠ 0):
             Input:  [gap, age_pred, (optionally MRI features)]
             Output: normal | premature | other
```

**Metrics reported:**
- Regression: MAE and RMSE between `age_pred` and `age` (age misestimation error)
- Classification: per-class accuracy, confusion matrix, and misclassification error for `premature` vs all other diagnoses

> ⚠️ **Runtime:** Set to `T4 GPU` via Runtime → Change runtime type → GPU

## 1. Install dependencies

In [2]:
!pip install -q nibabel nilearn zenodo-get scikit-learn monai einops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.3 MB/s eta 0:00:00


## 2. Download the dataset from Zenodo

In [3]:
import os

DATA_DIR = "/content/myelin_dataset"
os.makedirs(DATA_DIR, exist_ok=True)

# Download from Zenodo record 8055666
!zenodo_get 10.5281/zenodo.8055666 -o {DATA_DIR}

!ls {DATA_DIR} | head -20

INFO: Output directory: /content/myelin_dataset
INFO: Title: Large dataset of infancy and early childhood brain MRIs (T1w and T2w)
INFO: Total size: 12.1 GB
INFO: Number of files: 1
SUCCESS: All specified files have been processed.
zenodo_upload_v2.zip


## 3. Inspect and prepare metadata

In [15]:
import pandas as pd
import glob
import numpy as np
import os

# Check if the zip file exists and unzip it if it hasn't been already
zip_file_path = os.path.join(DATA_DIR, "zenodo_upload_v2.zip")
if os.path.exists(zip_file_path):
    print(f"Unzipping {zip_file_path}...")
    # Use -o to overwrite existing files, -q for quiet mode
    !unzip -o -q {zip_file_path} -d {DATA_DIR}
    print("Unzipping complete.")
else:
    print(f"Zip file not found at {zip_file_path}. Assuming data is already unzipped or not downloaded correctly.")

meta_path = glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)[0]
df = pd.read_csv(meta_path, sep=';') # Attempting to read with semicolon separator
df = df.dropna(subset=['age','age_corrected']).copy()
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Unzipping /content/myelin_dataset/zenodo_upload_v2.zip...
Unzipping complete.
Shape: (833, 7)
Columns: ['image_id', 'myelinisation', 'age', 'age_corrected', 'doctor_predicted_age', 'diagnosis', 'group']


,image_id,myelinisation,age,age_corrected,doctor_predicted_age,diagnosis,group
0,s0001,normal,22,22,22,normal,train
1,s0002,normal,29,29,29,normal,train
2,s0003,normal,4,4,4,normal,train
3,s0004,normal,0,0,0,normal,train
4,s0005,normal,0,0,0,normal,train


In [16]:

# GAP computation (correct definition)
df['gap'] = df['age'] - df['age_corrected']
df[['age','age_corrected','gap']].head()


,age,age_corrected,gap
0,22,22,0
1,29,29,0
2,4,4,0
3,0,0,0
4,0,0,0


In [18]:
# Inspect the diagnosis and age_pred columns
print("=== diagnosis distribution ===")
print(df['diagnosis'].value_counts())
print()
print("=== age_corrected sample ===")
print(df[['age', 'age_corrected']].describe())

=== diagnosis distribution ===
diagnosis
normal                      565
HIE                          45
Hydrocephalus                30
Infarct                      21
Meningitis                   17
                           ... 
focal cortical dysplasia      1
Crouzon syndrome              1
Sturge-Weber-Syndrom          1
cerebellar atrophy            1
medulloblastoma               1
Name: count, Length: 73, dtype: int64

=== age_corrected sample ===
              age  age_corrected
count  833.000000     833.000000
mean    13.722689      13.668667
std     11.754957      11.795154
min      0.000000      -1.000000
25%      3.000000       3.000000
50%     12.000000      12.000000
75%     24.000000      24.000000
max     36.000000      36.000000


## 4. Build file paths and filter subjects with both T1w + T2w

In [23]:
import os, glob

# Keep only subjects that have a valid age and age_pred
df = df.dropna(subset=['age', 'age_corrected']).copy()
print(f"Subjects with valid age/age_corrected: {len(df)}")

def find_nifti(subject_id, modality, data_dir):
    """Find a NIfTI file for a given subject and modality (t1 or t2)."""
    # Based on ls -R output, files are in <data_dir>/<subject_id>/<modality>.nii.gz
    path = os.path.join(data_dir, subject_id, f"{modality}.nii.gz")
    if os.path.exists(path):
        return path
    return None

# Detect the subject-id column
id_col = next(c for c in ['subject', 'subject_id', 'sub', 'id', 'SubjectID', 'image_id'] if c in df.columns)
print(f"Using ID column: {id_col}")

# --- Add this for debugging file structure ---
print(f"\nListing contents of {DATA_DIR} to debug NIfTI file paths:")
!ls -R {DATA_DIR}
print("-" * 50)
# ---------------------------------------------

t1_paths, t2_paths, valid_idx = [], [], []
for i, row in df.iterrows():
    sid = str(row[id_col])
    # Changed 'T1w' to 't1' and 'T2w' to 't2' based on file names
    t1 = find_nifti(sid, 't1', DATA_DIR)
    t2 = find_nifti(sid, 't2', DATA_DIR)
    if t1 and t2:
        t1_paths.append(t1)
        t2_paths.append(t2)
        valid_idx.append(i)

df_valid = df.loc[valid_idx].copy()
df_valid['t1_path'] = t1_paths
df_valid['t2_path'] = t2_paths
print(f"Subjects with both T1w + T2w found: {len(df_valid)}")

Subjects with valid age/age_corrected: 833
Using ID column: image_id

Listing contents of /content/myelin_dataset to debug NIfTI file paths:
/content/myelin_dataset:
atlas	  s0092  s0185	s0278  s0371  s0464  s0557  s0650  s0743
meta.csv  s0093  s0186	s0279  s0372  s0465  s0558  s0651  s0744
s0001	  s0094  s0187	s0280  s0373  s0466  s0559  s0652  s0745
s0002	  s0095  s0188	s0281  s0374  s0467  s0560  s0653  s0746
s0003	  s0096  s0189	s0282  s0375  s0468  s0561  s0654  s0747
s0004	  s0097  s0190	s0283  s0376  s0469  s0562  s0655  s0748
s0005	  s0098  s0191	s0284  s0377  s0470  s0563  s0656  s0749
s0006	  s0099  s0192	s0285  s0378  s0471  s0564  s0657  s0750
s0007	  s0100  s0193	s0286  s0379  s0472  s0565  s0658  s0751
s0008	  s0101  s0194	s0287  s0380  s0473  s0566  s0659  s0752
s0009	  s0102  s0195	s0288  s0381  s0474  s0567  s0660  s0753
s0010	  s0103  s0196	s0289  s0382  s0475  s0568  s0661  s0754
s0011	  s0104  s0197	s0290  s0383  s0476  s0569  s0662  s0755
s0012	  s0105  s0198	s0291

## 5. MONAI transforms — resize + normalise 3-D volumes

In [24]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd,
    Spacingd, Orientationd, ScaleIntensityRangePercentilesd,
    Resized, ConcatItemsd, ToTensord
)

TARGET_SHAPE = (64, 64, 64)   # Reduce if OOM; increase for better accuracy

mri_transforms = Compose([
    LoadImaged(keys=['t1', 't2']),
    EnsureChannelFirstd(keys=['t1', 't2']),
    Orientationd(keys=['t1', 't2'], axcodes='RAS'),
    Spacingd(keys=['t1', 't2'], pixdim=(2.0, 2.0, 2.0), mode='bilinear'),
    ScaleIntensityRangePercentilesd(keys=['t1', 't2'], lower=1, upper=99,
                                     b_min=0.0, b_max=1.0, clip=True),
    Resized(keys=['t1', 't2'], spatial_size=TARGET_SHAPE),
    ConcatItemsd(keys=['t1', 't2'], name='image', dim=0),  # → (2, D, H, W)
    ToTensord(keys=['image'])
])

/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


## 6. Dataset and DataLoaders — Step 1: Age Regression
Target: `age_pred` column (the expert-assigned or model-predicted brain age from the dataset).

In [27]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class AgeRegressionDataset(Dataset):
    """
    Returns (image, age_corrected) pairs.
    The model learns to predict age_corrected from raw MRI.
    """
    def __init__(self, dataframe, transforms):
        self.df = dataframe.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        data_dict = {'t1': row['t1_path'], 't2': row['t2_path']}
        data_dict = self.transforms(data_dict)
        image    = data_dict['image']                              # (2, D, H, W)
        age_pred = torch.tensor(row['age_corrected'], dtype=torch.float32)
        return image, age_pred

# 70/15/15 split (no stratification needed for regression)
train_df, test_df = train_test_split(df_valid, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(test_df,  test_size=0.50, random_state=42)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

BATCH = 4
train_reg_ds = AgeRegressionDataset(train_df, mri_transforms)
val_reg_ds   = AgeRegressionDataset(val_df,   mri_transforms)
test_reg_ds  = AgeRegressionDataset(test_df,  mri_transforms)

train_reg_loader = DataLoader(train_reg_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_reg_loader   = DataLoader(val_reg_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_reg_loader  = DataLoader(test_reg_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

Train: 583  Val: 125  Test: 125


## 7. Model — 3D ResNet-18 for Age Regression (1 output neuron)

In [28]:
import torch.nn as nn
from monai.networks.nets import resnet18

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class AgeRegressor(nn.Module):
    """ResNet-18 backbone with a single scalar regression head."""
    def __init__(self):
        super().__init__()
        self.backbone = resnet18(
            pretrained=False,
            spatial_dims=3,
            n_input_channels=2,   # T1w + T2w
            num_classes=1          # regression → scalar
        )
    def forward(self, x):
        return self.backbone(x).squeeze(1)  # → (B,)

reg_model = AgeRegressor().to(device)
print(f"Parameters: {sum(p.numel() for p in reg_model.parameters()):,}")

Using device: cpu
Parameters: 33,183,361


## 8. Loss, optimiser, scheduler — Regression

In [29]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

reg_criterion = nn.MSELoss()                           # MSE = RMSE² ; we also track MAE
reg_optimizer = AdamW(reg_model.parameters(), lr=1e-4, weight_decay=1e-5)
reg_scheduler = CosineAnnealingLR(reg_optimizer, T_max=30, eta_min=1e-6)

## 9. Training loop — Step 1: MRI → age_corrected

In [ ]:
EPOCHS_REG = 30
SAVE_REG   = "/content/best_age_regressor.pth"

reg_history = {'train_mse': [], 'val_mae': [], 'val_rmse': []}
best_val_mae = float('inf')

def run_reg_epoch(loader, training=True):
    reg_model.train() if training else reg_model.eval()
    total_mse = 0.0
    all_preds, all_targets = [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, targets in loader:
            images, targets = images.to(device), targets.to(device)
            preds = reg_model(images)
            loss  = reg_criterion(preds, targets)
            if training:
                reg_optimizer.zero_grad()
                loss.backward()
                reg_optimizer.step()
            total_mse  += loss.item() * len(targets)
            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    all_preds   = np.array(all_preds)
    all_targets = np.array(all_targets)
    mae  = np.mean(np.abs(all_preds - all_targets))
    rmse = np.sqrt(total_mse / len(loader.dataset))
    return mae, rmse, all_preds, all_targets

for epoch in range(1, EPOCHS_REG + 1):
    tr_mae, tr_rmse, _, _ = run_reg_epoch(train_reg_loader, training=True)
    vl_mae, vl_rmse, _, _ = run_reg_epoch(val_reg_loader,   training=False)
    reg_scheduler.step()

    reg_history['train_mse'].append(tr_rmse**2)
    reg_history['val_mae'].append(vl_mae)
    reg_history['val_rmse'].append(vl_rmse)

    if vl_mae < best_val_mae:
        best_val_mae = vl_mae
        torch.save(reg_model.state_dict(), SAVE_REG)
        flag = " ✓ saved"
    else:
        flag = ""

    print(f"Epoch {epoch:02d}/{EPOCHS_REG} | "
          f"Train RMSE {tr_rmse:.3f} | "
          f"Val MAE {vl_mae:.3f}  Val RMSE {vl_rmse:.3f}{flag}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 10. Training curves — Regression

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, EPOCHS_REG + 1)

axes[0].plot(ep, reg_history['train_mse'], label='Train MSE')
axes[0].set_title('Train MSE (age regression)'); axes[0].legend()

axes[1].plot(ep, reg_history['val_mae'],  label='Val MAE',  color='tab:orange')
axes[1].plot(ep, reg_history['val_rmse'], label='Val RMSE', color='tab:red')
axes[1].set_title('Validation Errors'); axes[1].legend()

plt.tight_layout()
plt.savefig('/content/reg_training_curves.png', dpi=120)
plt.show()

## 11. Step 2: Compute gap = age_pred − age on the test set
We reload the best regression model, run inference on the **full dataset** to collect model-predicted ages, then compute the brain-age gap.

In [ ]:
# ---- Reload best regressor ----
reg_model.load_state_dict(torch.load(SAVE_REG, map_location=device))
reg_model.eval()

# Run on full dataset to collect model-predicted ages for all subjects
# (needed to build the gap-based feature set for Step 3)
from torch.utils.data import Dataset, DataLoader

class FullDataset(Dataset):
    def __init__(self, dataframe, transforms):
        self.df = dataframe.reset_index(drop=True)
        self.transforms = transforms
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        d   = self.transforms({'t1': row['t1_path'], 't2': row['t2_path']})
        return d['image'], torch.tensor(row['age'], dtype=torch.float32)

full_loader = DataLoader(
    FullDataset(df_valid, mri_transforms),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True
)

model_age_preds, true_ages = [], []
with torch.no_grad():
    for images, ages in full_loader:
        preds = reg_model(images.to(device)).cpu().numpy()
        model_age_preds.extend(preds)
        true_ages.extend(ages.numpy())

model_age_preds = np.array(model_age_preds)
true_ages       = np.array(true_ages)
gaps            = model_age_preds - true_ages

df_valid = df_valid.copy()
df_valid['model_age_pred'] = model_age_preds  # our CNN's prediction
df_valid['gap']            = gaps

print("Gap statistics:")
print(pd.Series(gaps).describe())

# ---- Regression error between our model's age_pred and chronological age ----
mae_overall  = np.mean(np.abs(model_age_preds - true_ages))
rmse_overall = np.sqrt(np.mean((model_age_preds - true_ages)**2))
print(f"\n=== Age Prediction Error (Step 1) ===")
print(f"  MAE  (age_pred vs age): {mae_overall:.4f}")
print(f"  RMSE (age_pred vs age): {rmse_overall:.4f}")

In [ ]:
# Visualise the gap distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(gaps, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', label='gap = 0')
axes[0].set_xlabel('gap = model_age_pred − true_age')
axes[0].set_title('Distribution of Brain-Age Gap')
axes[0].legend()

axes[1].scatter(true_ages, model_age_preds, alpha=0.5, s=15, c=gaps,
                cmap='coolwarm', vmin=-np.percentile(np.abs(gaps), 95),
                vmax=np.percentile(np.abs(gaps), 95))
axes[1].plot([true_ages.min(), true_ages.max()],
             [true_ages.min(), true_ages.max()], 'k--', label='perfect fit')
axes[1].set_xlabel('True Age'); axes[1].set_ylabel('Predicted Age')
axes[1].set_title('Age Prediction Scatter')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/gap_analysis.png', dpi=120)
plt.show()

## 12. Step 3: Assign 3-class diagnosis labels
```
normal    → diagnosis == 'normal'  AND  gap ≈ 0
premature → 'preterm' in diagnosis column
other     → everything else (polymicrogyria, HIE, etc.)
```

In [ ]:
def assign_3class_label(row):
    """
    Returns:
      0 = normal
      1 = premature  (diagnosis contains 'preterm')
      2 = other      (any pathological diagnosis that is not preterm)
    """
    diag = str(row.get('diagnosis', '')).lower().strip()
    if 'preterm' in diag:
        return 1   # premature
    elif diag == 'normal':
        return 0   # normal
    else:
        return 2   # other pathology (HIE, polymicrogyria, …)

df_valid['label3'] = df_valid.apply(assign_3class_label, axis=1)

label_names = {0: 'normal', 1: 'premature', 2: 'other'}
print("3-class label distribution:")
print(df_valid['label3'].map(label_names).value_counts())

## 13. Dataset and DataLoader — Step 3: Diagnosis Classifier
Input features: **[gap, model_age_pred]** + a flattened global-average-pooled MRI feature vector extracted from the frozen regression backbone.

In [ ]:
# ── Extract global-average-pooled MRI features from the frozen regressor ──
# We hook into the layer just before the final FC head.

reg_model.eval()
mri_features_list = []

def hook_fn(module, input, output):
    # output shape: (B, C, 1, 1, 1)  after adaptive avg pool
    mri_features_list.append(output.detach().cpu().squeeze(-1).squeeze(-1).squeeze(-1))  # (B, C)

# Register hook on the global average pool layer of the MONAI resnet18 backbone
hook = reg_model.backbone.avgpool.register_forward_hook(hook_fn)

feature_loader = DataLoader(
    FullDataset(df_valid, mri_transforms),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True
)

with torch.no_grad():
    for images, _ in feature_loader:
        reg_model(images.to(device))  # triggers hook

hook.remove()
mri_feats = torch.cat(mri_features_list, dim=0).numpy()  # (N, C)
print(f"MRI feature matrix shape: {mri_feats.shape}")

In [ ]:
# Combine tabular features [gap, model_age_pred] with MRI embedding
gap_feat       = df_valid['gap'].values.reshape(-1, 1)
age_pred_feat  = df_valid['model_age_pred'].values.reshape(-1, 1)
X_all          = np.hstack([gap_feat, age_pred_feat, mri_feats])  # (N, 2 + C)
y_all          = df_valid['label3'].values

print(f"Feature matrix: {X_all.shape}  |  Labels: {y_all.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split to preserve class proportions
X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=0.25, stratify=y_all, random_state=42
)
print(f"Classifier train: {X_tr.shape[0]}  |  test: {X_te.shape[0]}")

# Normalise feature values
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

## 14. Train the 3-class Diagnosis Classifier (MLP + optional Random Forest)

In [ ]:
# ── Option A: lightweight MLP (fast, no GPU needed at this stage) ──
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20
)
mlp.fit(X_tr_s, y_tr)

y_pred_mlp = mlp.predict(X_te_s)
print("=== 3-class Diagnosis Classifier (MLP) ===")
print(classification_report(y_te, y_pred_mlp, target_names=['normal', 'premature', 'other']))

In [ ]:
# ── Option B: Random Forest (often better with small N) ──
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_tr_s, y_tr)

y_pred_rf = rf.predict(X_te_s)
print("=== 3-class Diagnosis Classifier (Random Forest) ===")
print(classification_report(y_te, y_pred_rf, target_names=['normal', 'premature', 'other']))

## 15. Misclassification Errors

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    balanced_accuracy_score, roc_auc_score
)

# Choose best classifier (replace y_pred with y_pred_rf if RF is better)
y_pred = y_pred_rf

# ── Error 1: Age prediction error (Step 1) ──
print("═" * 55)
print(" ERROR 1: Age Prediction (MRI → age_pred vs true age)")
print("═" * 55)
print(f"  MAE  : {mae_overall:.4f}  (mean absolute error in same units as age)")
print(f"  RMSE : {rmse_overall:.4f}")

# ── Error 2: 3-class diagnosis misclassification ──
total       = len(y_te)
n_wrong     = (y_pred != y_te).sum()
overall_err = n_wrong / total

print()
print("═" * 55)
print(" ERROR 2: Diagnosis Classification (3-class overall)")
print("═" * 55)
print(f"  Overall misclassification rate : {overall_err:.4f}  ({n_wrong}/{total})")

# ── Error 3: premature vs. other (binary confusion) ──
# Subjects that are either premature (1) or other pathology (2)
mask_non_normal = y_te != 0
y_te_nn   = y_te[mask_non_normal]
y_pred_nn = y_pred[mask_non_normal]

# Relabel: premature=1, other=0
y_te_bin   = (y_te_nn   == 1).astype(int)
y_pred_bin = (y_pred_nn == 1).astype(int)

n_wrong_bin  = (y_pred_bin != y_te_bin).sum()
err_prem_vs_other = n_wrong_bin / len(y_te_bin) if len(y_te_bin) > 0 else float('nan')

print()
print("═" * 55)
print(" ERROR 3: premature vs. other (non-normal subjects only)")
print("═" * 55)
print(f"  Misclassification rate : {err_prem_vs_other:.4f}  ({n_wrong_bin}/{len(y_te_bin)})")

# Per-class breakdown
print()
print("Per-class breakdown (premature vs other):")
print(classification_report(y_te_bin, y_pred_bin, target_names=['other', 'premature']))

## 16. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 3-class confusion matrix ──
cm3 = confusion_matrix(y_te, y_pred)
sns.heatmap(cm3, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['normal', 'premature', 'other'],
            yticklabels=['normal', 'premature', 'other'])
axes[0].set_title('3-class Confusion Matrix')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

# ── premature vs other confusion matrix ──
cm2 = confusion_matrix(y_te_bin, y_pred_bin)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['other', 'premature'],
            yticklabels=['other', 'premature'])
axes[1].set_title('Premature vs Other Confusion Matrix')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('/content/confusion_matrices.png', dpi=120)
plt.show()

## 17. Gap Analysis per Diagnosis Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box-plot of gap by 3-class label
groups     = [df_valid.loc[df_valid['label3'] == c, 'gap'].values for c in [0, 1, 2]]
axes[0].boxplot(groups, labels=['normal', 'premature', 'other'], patch_artist=True)
axes[0].axhline(0, color='red', linestyle='--', label='gap = 0')
axes[0].set_title('Brain-Age Gap by Diagnosis')
axes[0].set_ylabel('gap = model_age_pred − true_age')
axes[0].legend()

# Violin plot
import matplotlib.patches as mpatches
colors = ['#4CAF50', '#2196F3', '#FF5722']
parts  = axes[1].violinplot(groups, positions=[1, 2, 3], showmedians=True)
for body, c in zip(parts['bodies'], colors):
    body.set_facecolor(c); body.set_alpha(0.7)
axes[1].set_xticks([1, 2, 3])
axes[1].set_xticklabels(['normal', 'premature', 'other'])
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Gap Distribution per Class (Violin)')
axes[1].set_ylabel('gap')

plt.tight_layout()
plt.savefig('/content/gap_per_class.png', dpi=120)
plt.show()

---
## Summary of Reported Metrics

| Error | Metric | Description |
|-------|--------|-------------|
| Age prediction | MAE, RMSE | Difference between CNN's `model_age_pred` and true `age` |
| 3-class diagnosis | Overall misclassification rate | Fraction of test subjects wrongly labelled normal/premature/other |
| Premature vs other | Binary misclassification rate | Among non-normal subjects only: premature confused with other pathology |

## Notes
- **Dataset download** takes ~10–20 min; the ~833 NIfTI pairs are several GB total.
- **`TARGET_SHAPE`** is `(64,64,64)`. Raise to `(96,96,96)` for better accuracy at the cost of GPU memory.
- **3-class imbalance**: weighted `class_weight='balanced'` is applied in the Random Forest; for the MLP you may add `class_weight` via a custom loss if needed.
- **`diagnosis` column**: the label assignment in cell 12 assumes the column contains strings like `'normal'`, `'preterm'`, `'HIE'`, `'polymicrogyria'`, etc. Adjust the `assign_3class_label` function if your column uses different naming.
- **Cite the dataset**: Akinci D'Antonoli et al., Radiology: AI, https://pubs.rsna.org/doi/10.1148/ryai.220292